# LAG_PAIRS 検証

JP 波及タイミング予測の根幹となる「EN→JP ラグ推定ロジック」が正しく機能するかを、
過去に波及が確認済みの確定 LAG_PAIRS（4ペア）を使って検証する。

## 確定 LAG_PAIRS（クラスターA）

| EN キーワード | JP キーワード | 既知ラグ |
|---|---|---|
| Y2K fashion | Y2Kファッション | 11ヶ月 |
| balletcore | バレエコア | 14ヶ月 |
| low rise jeans | ローライズ | 16ヶ月 |
| normcore | ノームコア | 14ヶ月 |

## 検証の構成

1. **クロスコリレーション** — EN/JP 系列を時間方向にずらしながら相関を計算し、ピーク位置が既知ラグと一致するか確認する。単純な時間差モデルとしての有効性を評価。
2. **DTW アナログマッチング（セルフマッチ確認）** — 各ペアの EN 波形を `dtw_analog_match` に通し、自分自身が「最も近いテンプレート（距離=0）」として返ってくるかを確認する。波形抽出・距離計算が正しく機能しているかのサニティチェック。
3. **EN 波形比較** — 4ペアのピーク前 24 ヶ月の EN 波形を正規化して重ね描きし、DTW マッチングの入力データを可視化する。

In [1]:
import sys
sys.path.append('..')

import logging
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from trend_utils import load_category, get_peak, EN_CATS, JP_CATS, PERIODS
from lag_estimator import LAG_PAIRS, cross_correlate_lag, dtw_analog_match, AVG_LAG_MONTHS
from synonym_groups import SYNONYM_GROUPS

## 0. 確定ラグの算出

Google Trends の EN/JP 時系列それぞれに対して `get_peak` でピーク日を特定し、
JP peak date − EN peak date を月数に換算してラグを算出する。

この値が `lag_estimator.py` の `LAG_PAIRS` にハードコードされている根拠となる。

**ペア選定・期間選択について**  
過去10年分のデータから EN→JP 波及が明確に確認できるペアを候補として調査し、
「ENピーク後に対応するJPピークが存在する」ことをグラフで目視確認した上で確定した。
複数波がある場合は第一波（最初の本格的な波及）を採用するなど、一部手動での判断を含む。  
→ 選定過程の詳細は [`docs/lag_pairs_analysis.md`](../docs/lag_pairs_analysis.md) を参照。

> **注**: 下表のラグは全期間・全カテゴリを自動走査した近似値。キーワードによってはノイズ系列を拾うため実際の値とずれる場合がある。確定ペアの正確なラグは Section 1 以降で確認すること。

In [2]:
def find_en_series(keyword):
    """全EN期間・カテゴリを検索し、ピーク日が最も早い系列を返す（最初の波及を捉える）。"""
    best_series, best_peak = None, None
    for period in PERIODS:
        for cat in EN_CATS:
            df = load_category(cat, period)
            if keyword not in df.columns:
                continue
            s = df[keyword]
            _, peak_date = get_peak(s)
            if best_peak is None or peak_date < best_peak:
                best_series, best_peak = s, peak_date
    return best_series


def find_jp_series(keyword, en_peak_date):
    """全JP期間・カテゴリを検索し、ENピーク後で最初にピークを打つ系列を返す。
    期間末尾がピークの系列（データ打ち切りアーティファクト）はスキップ。"""
    best_series, best_peak = None, None
    for period in PERIODS:
        for cat in JP_CATS:
            df = load_category(cat, period)
            if keyword not in df.columns:
                continue
            s = df[keyword]
            _, peak_date = get_peak(s)
            if peak_date == s.index[-1]:
                continue
            if peak_date <= en_peak_date:
                continue
            if best_peak is None or peak_date < best_peak:
                best_series, best_peak = s, peak_date
    return best_series


# 全候補ペアのEN→JPラグを算出
# 選定根拠の詳細は docs/lag_pairs_analysis.md を参照。

rows = []
series_store = {}

for g in SYNONYM_GROUPS:
    en_kw = g['en'][0]
    jp_kw = g['jp_ja'][0] if g['jp_ja'] else None
    if not jp_kw:
        continue

    en_series = find_en_series(en_kw)
    if en_series is None:
        rows.append({'status': g['status'], 'EN keyword': en_kw, 'JP keyword': jp_kw,
                     'EN peak': '—', 'JP peak': '—', 'ラグ（ヶ月）': None})
        continue

    _, en_peak_date = get_peak(en_series)
    jp_series = find_jp_series(jp_kw, en_peak_date)

    if jp_series is None:
        rows.append({'status': g['status'], 'EN keyword': en_kw, 'JP keyword': jp_kw,
                     'EN peak': en_peak_date.strftime('%Y-%m'), 'JP peak': '—', 'ラグ（ヶ月）': None})
        continue

    _, jp_peak_date = get_peak(jp_series)
    lag = month_diff(en_peak_date, jp_peak_date)

    series_store[en_kw] = {
        'en': en_series, 'jp': jp_series,
        'en_peak': en_peak_date, 'jp_peak': jp_peak_date,
        'jp_kw': jp_kw,
    }
    rows.append({
        'status': g['status'],
        'EN keyword': en_kw,
        'JP keyword': jp_kw,
        'EN peak': en_peak_date.strftime('%Y-%m'),
        'JP peak': jp_peak_date.strftime('%Y-%m'),
        'ラグ（ヶ月）': lag,
    })

# Section 1以降のチャート用に confirmed ペアを LAG_PAIRS の正確なデータで上書き
for pair in LAG_PAIRS:
    en_s = load_category(pair['en_cat'], pair['en_period'])[pair['en_kw']]
    jp_s = load_category(pair['jp_cat'], pair['jp_period'])[pair['jp_kw']]
    _, en_peak = get_peak(en_s)
    _, jp_peak = get_peak(jp_s)
    series_store[pair['en_kw']] = {
        'en': en_s, 'jp': jp_s,
        'en_peak': en_peak, 'jp_peak': jp_peak,
        'jp_kw': pair['jp_kw'],
    }

status_order = ['confirmed', 'cluster_b', 'cluster_c', 'monitoring', 'excluded', 'jp_leads']
df = pd.DataFrame(rows)
df['status'] = pd.Categorical(df['status'], categories=status_order, ordered=True)
df.sort_values('status').reset_index(drop=True)

,status,EN keyword,JP keyword,EN peak,JP peak,ラグ（ヶ月）
0,confirmed,Y2K fashion,Y2Kファッション,2021-01,2022-08,19.0
1,confirmed,balletcore,バレエコア,2016-04,2024-09,101.0
2,confirmed,low rise jeans,ローライズ,2018-12,2020-04,16.0
3,confirmed,normcore,ノームコア,2018-03,2019-05,14.0
4,cluster_b,mermaid skirt,マーメイドスカート,2016-10,2021-02,52.0
5,cluster_c,barrel jeans,バレルジーンズ,2024-02,2024-07,5.0
6,monitoring,brooch aesthetic,ブローチコーデ,2026-02,—,NaN
7,monitoring,lace outfit,レースコーデ,2026-02,—,NaN
8,monitoring,glitchy glam,メタリックコーデ,2021-12,2024-02,26.0
9,monitoring,extra celestial aesthetic,オパールコーデ,2025-06,—,NaN


In [3]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[p['label'] for p in LAG_PAIRS],
    vertical_spacing=0.18, horizontal_spacing=0.1,
)

for i, pair in enumerate(LAG_PAIRS):
    row, col = i // 2 + 1, i % 2 + 1
    st = series_store[pair['en_kw']]

    # EN 系列（青）
    fig.add_trace(go.Scatter(
        x=st['en'].index, y=st['en'].values,
        name='EN', line=dict(color='#4472C4', width=2),
        showlegend=(i == 0),
    ), row=row, col=col)

    # JP 系列（紫点線）
    fig.add_trace(go.Scatter(
        x=st['jp'].index, y=st['jp'].values,
        name='JP', line=dict(color='#7B68EE', width=2, dash='dot'),
        showlegend=(i == 0),
    ), row=row, col=col)

    # EN ピーク（緑縦線）
    fig.add_vline(
        x=st['en_peak'].timestamp() * 1000,
        line=dict(color='#2CA02C', dash='dash', width=1.5),
        row=row, col=col,
    )

    # JP ピーク（赤縦線）
    fig.add_vline(
        x=st['jp_peak'].timestamp() * 1000,
        line=dict(color='#E15759', dash='dash', width=1.5),
        row=row, col=col,
    )

    lag = (st['jp_peak'].year - st['en_peak'].year) * 12 + (st['jp_peak'].month - st['en_peak'].month)
    fig.add_annotation(
        x=st['jp_peak'], y=st['jp'].max() * 0.85,
        text=f'+{lag}ヶ月',
        showarrow=False, font=dict(size=11, color='#E15759'),
        row=row, col=col,
    )

fig.update_layout(
    title=dict(text='確定 LAG_PAIRS — EN（青）/ JP（紫点線）時系列と各ピーク（緑: EN / 赤: JP）', font=dict(size=14)),
    height=600, template='plotly_white',
)
fig.show()

## 1. クロスコリレーション検証

EN 系列と JP 系列のラグ（0〜30ヶ月）を 1ヶ月ずつずらしながらピアソン相関を計算し、
相関が最大になるラグ（紫破線）が既知ラグ（赤点線）と一致するかを確認する。

**グラフの見方**
- 横軸: EN に対して JP を何ヶ月後ろにずらしたか
- 縦軸: そのずらし幅での EN/JP ピアソン相関係数
- 赤点線: 既知ラグ（正解）
- 紫破線: クロスコリレーションが最大になるラグ（推定値）

**結果と考察**

| ペア | 既知ラグ | 推定ラグ | 最大相関 | 評価 |
|---|---|---|---|---|
| balletcore → バレエコア | 14ヶ月 | 15ヶ月 | 0.646 | ✅ ほぼ一致 |
| Y2K fashion → Y2Kファッション | 11ヶ月 | 0ヶ月 | 0.396 | ❌ ズレ大 |
| normcore → ノームコア | 14ヶ月 | 29ヶ月 | 0.149 | ❌ ズレ大 |
| low rise jeans → ローライズ | 16ヶ月 | 19ヶ月 | -0.066 | ❌ 相関マイナス |

4ペア中 1ペアしか正確に推定できなかった。クロスコリレーションは EN/JP が「形状は同じだが時間差だけある」シンプルな波形には有効だが、
ファッショントレンドのように JP 側が EN より緩やかに立ち上がったり、ピーク形状が異なるケースでは機能しにくい。

→ **波形の "形の類似度" を柔軟に扱える DTW アナログマッチングを採用する根拠となった。**

In [4]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[p['label'] for p in LAG_PAIRS],
    vertical_spacing=0.18, horizontal_spacing=0.1,
)

results_cc = []
for i, pair in enumerate(LAG_PAIRS):
    row, col = i // 2 + 1, i % 2 + 1
    st = series_store[pair['en_kw']]
    en_series = st['en']
    jp_series = st['jp']

    best_lag, best_corr, corr_curve = cross_correlate_lag(en_series, jp_series)
    results_cc.append({
        'label': pair['label'], 'known_lag': pair['lag_months'],
        'estimated_lag': best_lag, 'corr': round(best_corr, 3),
    })

    lags = list(range(len(corr_curve)))
    fig.add_trace(go.Scatter(
        x=lags, y=corr_curve,
        line=dict(color='#4472C4', width=2), showlegend=False,
    ), row=row, col=col)
    fig.add_vline(x=best_lag, line=dict(color='#E76BF3', dash='dash', width=2),
                  row=row, col=col)
    fig.add_vline(x=pair['lag_months'], line=dict(color='#E15759', dash='dot', width=2),
                  row=row, col=col)
    fig.add_annotation(
        x=best_lag, y=best_corr,
        text=f"推定: {best_lag}ヶ月<br>既知: {pair['lag_months']}ヶ月",
        showarrow=True, arrowhead=2, font=dict(size=10),
        row=row, col=col,
    )

fig.update_xaxes(title_text='ラグ（ヶ月）')
fig.update_yaxes(title_text='相関係数')
fig.update_layout(
    title=dict(text='クロスコリレーション — ラグ vs 相関（紫破線: 推定 / 赤点線: 既知）', font=dict(size=14)),
    height=600, template='plotly_white',
)
fig.show()

pd.DataFrame(results_cc)

,label,known_lag,estimated_lag,corr
0,Y2K fashion → Y2Kファッション,11,0,0.396
1,balletcore → バレエコア,14,15,0.646
2,low rise jeans → ローライズ,16,19,-0.066
3,normcore → ノームコア,14,29,0.149


## 2. DTW アナログマッチング — セルフマッチ確認

**DTW アナログマッチングとは**

未知キーワードの EN 波形（ピーク前 24 ヶ月・正規化）を、確定 LAG_PAIRS 4本の波形と DTW で照合し、
最も波形が近いペアのラグを「そのキーワードの EN→JP ラグ」として採用する手法。

DTW（Dynamic Time Warping）は波形の伸縮を許容しながら類似度を計算するため、
ピークの急峻さや立ち上がりのタイミングが多少異なっても正しく対応付けられる。

**検証内容**

各 LAG_PAIRS の EN 波形を `dtw_analog_match` に入力し、
**自分自身が「最近傍テンプレート（DTW距離=0）」として返ってくるか**を確認する。
距離=0 は「完全に同一の波形」を意味し、波形抽出と距離計算が正しく動作していることのサニティチェックになる。

続くヒートマップは 4×4 の DTW 距離マトリックスで、各行が「そのキーワードを未知として入力したとき」の
全テンプレートとの距離を示す。対角成分（自己距離）が最小になることを視覚的に確認できる。

**結果**

全 4 ペアで DTW距離=0.0、推定ラグ=既知ラグと完全一致。
波形抽出・正規化・DTW 距離計算が正しく実装されていることを確認。

In [5]:
results_dtw = []
for pair in LAG_PAIRS:
    en_series = load_category(pair['en_cat'], pair['en_period'])[pair['en_kw']]
    _, en_peak_date = get_peak(en_series)
    match = dtw_analog_match(en_series, en_peak_date)
    results_dtw.append({
        'キーワード':  pair['en_kw'],
        '既知ラグ':   pair['lag_months'],
        '最近傍ペア': match['best_template'],
        '推定ラグ':   match['lag_months'],
        'DTW距離':    round(match['dtw_distance'], 3),
        '全距離':     match['all_distances'],
    })

df_dtw = pd.DataFrame(results_dtw)
df_dtw[['キーワード', '既知ラグ', '最近傍ペア', '推定ラグ', 'DTW距離']]

,キーワード,既知ラグ,最近傍ペア,推定ラグ,DTW距離
0,Y2K fashion,11,Y2K fashion → Y2Kファッション,11,0.0
1,balletcore,14,balletcore → バレエコア,14,0.0
2,low rise jeans,16,low rise jeans → ローライズ,16,0.0
3,normcore,14,normcore → ノームコア,14,0.0


In [6]:
# DTW 距離ヒートマップ（類似度マトリックス）
labels = [p['en_kw'] for p in LAG_PAIRS]
matrix = np.zeros((len(LAG_PAIRS), len(LAG_PAIRS)))
for i, pair_i in enumerate(LAG_PAIRS):
    s_i = load_category(pair_i['en_cat'], pair_i['en_period'])[pair_i['en_kw']]
    _, peak_i = get_peak(s_i)
    match_i = dtw_analog_match(s_i, peak_i)
    for j, pair_j in enumerate(LAG_PAIRS):
        matrix[i, j] = match_i['all_distances'].get(pair_j['label'], 0)

fig_heat = go.Figure(go.Heatmap(
    z=matrix, x=labels, y=labels,
    colorscale='Blues_r', text=np.round(matrix, 2),
    texttemplate='%{text}', showscale=True,
))
fig_heat.update_layout(
    title='DTW 距離マトリックス（小さいほど波形が類似）',
    height=400, template='plotly_white',
)
fig_heat.show()

## 3. EN 波形比較（確定ペア）

**このグラフの目的**

DTW アナログマッチングは「波形の形の類似度」でテンプレートを選ぶ。
そのため「4本のテンプレートが互いに形が違うこと」が重要で、似たような形のテンプレートが複数あると
どのキーワードにも同じラグが当たってしまい、ラグ推定の意味が薄れる。

ここでは 4 本の EN 波形を**ピーク=1 に正規化し横軸をピークまでの月数に揃えて**重ね描きすることで、
テンプレートの形状多様性を確認する。

**グラフの見方**
- 横軸: ピーク時点（x=0）を基準とした月数（マイナス方向 = ピーク前）
- 縦軸: 正規化スコア（ピーク=1.0）
- 各線の形が「そのキーワードの流行の立ち上がり方」を示す

**観察と考察**

4ペアは取得期間も geo もバラバラ（2016-2021 / 2021-2026、styles / items）だが、
いずれも「緩やかな助走 → ピークへの急上昇」という共通のパターンを持つ。
一方でピークまでの立ち上がり速度や肩の形には違いがあり、テンプレートとして識別可能な多様性が確認できる。

新規キーワードの波形がどのテンプレートに最も近いかによって 11〜16ヶ月の範囲でラグを選択でき、
平均値（13.8ヶ月）を使うより精度が上がる設計になっている。

In [7]:
colors = ['#4472C4', '#E15759', '#F28E2B', '#76B7B2']
fig_wave = go.Figure()

for pair, color in zip(LAG_PAIRS, colors):
    s = load_category(pair['en_cat'], pair['en_period'])[pair['en_kw']]
    _, peak_date = get_peak(s)
    start = peak_date - pd.DateOffset(months=24)
    wave = s[(s.index >= start) & (s.index <= peak_date)].fillna(0)
    peak_val = wave.max()
    norm_wave = wave / peak_val if peak_val > 0 else wave
    x_months = list(range(-len(norm_wave) + 1, 1))
    fig_wave.add_trace(go.Scatter(
        x=x_months, y=norm_wave.values,
        name=f"{pair['en_kw']} (lag={pair['lag_months']}M)",
        line=dict(color=color, width=2),
    ))

fig_wave.add_vline(x=0, line=dict(color='black', dash='dash', width=1))
fig_wave.update_layout(
    title='確定ペア EN 波形（ピーク前 24 ヶ月・正規化）',
    xaxis_title='ピークまでの月数', yaxis_title='正規化スコア',
    height=400, template='plotly_white',
)
fig_wave.show()